# Tratamento de dados dos sensores — VulcanTech Smart Metals

Este notebook prepara os dados industriais para a dashboard e para o treinamento do modelo. Execute as células na ordem, ou use **Executar tudo**. Os arquivos finais são salvos em `saida/` e `site/data/`.

## 1. Bibliotecas e caminhos

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

RAIZ = Path.cwd().resolve()
if not (RAIZ / 'dados_sensores_vulcantech.csv').exists():
    RAIZ = RAIZ.parent

ARQUIVO = RAIZ / 'dados_sensores_vulcantech.csv'
PASTA_SAIDA = RAIZ / 'saida'
PASTA_SITE = RAIZ / 'site' / 'data'
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
PASTA_SITE.mkdir(parents=True, exist_ok=True)

LIMITES = {
    'temperatura_fuso_c': (30, 75),
    'vibracao_triaxial_rms_mms': (0.8, 3.8),
    'torque_fuso_nm': (45, 210),
    'vazao_refrigerante_lmin': (15, 40),
    'corrente_estator_amp': (12, 55),
    'pressao_hidraulica_bar': (110, 220),
    'nivel_ruido_db': (45, 85),
}

if not ARQUIVO.exists():
    raise FileNotFoundError(f'CSV original não encontrado: {ARQUIVO}')
print(f'Arquivo de entrada: {ARQUIVO}')

Arquivo de entrada: C:\Users\gabriel_l_simao\Documents\GitHub\VulcanTech_Smart_Metals\dados_sensores_vulcantech.csv


## 2. Leitura e diagnóstico inicial

Os dados são ordenados por horário antes da preparação, preservando a sequência temporal usada na avaliação do modelo.

In [2]:
dados = pd.read_csv(ARQUIVO, parse_dates=['timestamp'])
dados = dados.sort_values(['timestamp', 'id_celula']).reset_index(drop=True)
sensores = list(LIMITES)
print(f'{len(dados):,} registros · {len(dados.columns)} colunas')
display(dados.head())
display(dados[sensores].isna().sum().rename('leituras ausentes').to_frame())

10,000 registros · 11 colunas


,timestamp,id_celula,liga_metalica,temperatura_fuso_c,vibracao_triaxial_rms_mms,torque_fuso_nm,vazao_refrigerante_lmin,corrente_estator_amp,pressao_hidraulica_bar,nivel_ruido_db,tipo_falha
0,2026-10-01 00:00:00,CEL-202,Aco_Carbono_316L,56.04,NaN,95.68,26.31,23.73,168.79,53.96,0
1,2026-10-01 00:05:00,CEL-203,Aco_Carbono_316L,50.76,1.93,118.89,29.97,28.31,144.27,53.43,0
2,2026-10-01 00:10:00,CEL-203,Aco_Carbono_316L,54.95,2.09,96.19,29.16,36.11,159.42,54.23,0
3,2026-10-01 00:15:00,CEL-201,Aco_Carbono_316L,58.18,1.90,101.89,29.31,27.77,162.28,52.93,0
4,2026-10-01 00:20:00,CEL-202,Liga_Titanio_Ti6Al4V,65.81,1.82,132.07,27.21,39.15,147.51,73.67,0


,leituras ausentes
temperatura_fuso_c,250
vibracao_triaxial_rms_mms,250
torque_fuso_nm,0
vazao_refrigerante_lmin,200
corrente_estator_amp,0
pressao_hidraulica_bar,0
nivel_ruido_db,0


## 3. Ausências por célula e liga

Este resumo registra a proporção de valores ausentes antes da correção de sentinelas e ruído.

In [3]:
faltantes = (
    dados.groupby(['id_celula', 'liga_metalica'])[sensores]
    .apply(lambda grupo: grupo.isna().mean().mul(100))
    .round(2)
)
faltantes.to_csv(PASTA_SAIDA / 'faltantes_por_celula_liga.csv')
display(faltantes.head(10))

temperatura_fuso_c  vibracao_triaxial_rms_mms  \
id_celula liga_metalica                                                         
CEL-201   Aco_Carbono_316L                    2.28                       2.20   
          Aluminio_7075                       3.31                       1.78   
          Liga_Titanio_Ti6Al4V                2.05                       2.46   
CEL-202   Aco_Carbono_316L                    3.66                       2.49   
          Aluminio_7075                       1.91                       2.29   
          Liga_Titanio_Ti6Al4V                1.96                       2.54   
CEL-203   Aco_Carbono_316L                    2.67                       3.24   
          Aluminio_7075                       1.55                       2.26   
          Liga_Titanio_Ti6Al4V                2.62                       2.21   
CEL-204   Aco_Carbono_316L                    2.53                       2.61   

                                torque_fuso_nm  vazao_refrigerante_lmin  \
id_celula liga_metalica                                                   
CEL-201   Aco_Carbono_316L                 0.0                     2.03   
          Aluminio_7075                    0.0                     1.53   
          Liga_Titanio_Ti6Al4V             0.0                     2.05   
CEL-202   Aco_Carbono_316L                 0.0                     2.57   
          Aluminio_7075                    0.0                     1.65   
          Liga_Titanio_Ti6Al4V             0.0                     1.96   
CEL-203   Aco_Carbono_316L                 0.0                     1.13   
          Aluminio_7075                    0.0                     1.69   
          Liga_Titanio_Ti6Al4V             0.0                     1.01   
CEL-204   Aco_Carbono_316L                 0.0                     3.43   

                                corrente_estator_amp  pressao_hidraulica_bar  \
id_celula liga_metalica                                                        
CEL-201   Aco_Carbono_316L                       0.0                     0.0   
          Aluminio_7075                          0.0                     0.0   
          Liga_Titanio_Ti6Al4V                   0.0                     0.0   
CEL-202   Aco_Carbono_316L                       0.0                     0.0   
          Aluminio_7075                          0.0                     0.0   
          Liga_Titanio_Ti6Al4V                   0.0                     0.0   
CEL-203   Aco_Carbono_316L                       0.0                     0.0   
          Aluminio_7075                          0.0                     0.0   
          Liga_Titanio_Ti6Al4V                   0.0                     0.0   
CEL-204   Aco_Carbono_316L                       0.0                     0.0   

                                nivel_ruido_db  
id_celula liga_metalica                         
CEL-201   Aco_Carbono_316L                 0.0  
          Aluminio_7075                    0.0  
          Liga_Titanio_Ti6Al4V             0.0  
CEL-202   Aco_Carbono_316L                 0.0  
          Aluminio_7075                    0.0  
          Liga_Titanio_Ti6Al4V             0.0  
CEL-203   Aco_Carbono_316L                 0.0  
          Aluminio_7075                    0.0  
          Liga_Titanio_Ti6Al4V             0.0  
CEL-204   Aco_Carbono_316L                 0.0

## 4. Limites físicos e valores extremos

Leituras fora dos limites definidos são marcadas como ausentes. Também removemos valores com Z-score acima de 4,5 dentro de cada liga.

In [4]:
for coluna, (minimo, maximo) in LIMITES.items():
    fora = ~dados[coluna].between(minimo, maximo) & dados[coluna].notna()
    dados.loc[fora, coluna] = np.nan

    media = dados.groupby('liga_metalica')[coluna].transform('mean')
    desvio = dados.groupby('liga_metalica')[coluna].transform('std').replace(0, np.nan)
    z_score = (dados[coluna] - media).abs() / desvio
    dados.loc[z_score > 4.5, coluna] = np.nan

print('Limites físicos e valores extremos tratados.')

Limites físicos e valores extremos tratados.


## 5. Preenchimento temporal

A interpolação ocorre dentro de cada célula, usando o timestamp. Assim, leituras de células diferentes não são misturadas.

In [5]:
for coluna in sensores:
    for indices in dados.groupby('id_celula', sort=False).groups.values():
        tempos = pd.DatetimeIndex(dados.loc[indices, 'timestamp'])
        serie = pd.Series(dados.loc[indices, coluna].to_numpy(), index=tempos)
        preenchida = serie.interpolate(method='time', limit_direction='both').ffill().bfill()
        dados.loc[indices, coluna] = preenchida.to_numpy()

print('Preenchimento temporal concluído.')

Preenchimento temporal concluído.


## 6. Variáveis de processo

Criamos médias e desvios móveis de 30 minutos e 2 horas, o indicador IEM e razões relativas à média de cada liga.

In [6]:
janelas = {'30m': 6, '2h': 24}
grupos = dados.groupby('id_celula', sort=False)
dinamicas = [
    'temperatura_fuso_c', 'vibracao_triaxial_rms_mms',
    'torque_fuso_nm', 'vazao_refrigerante_lmin',
]

for nome, amostras in janelas.items():
    for coluna in dinamicas:
        dados[f'{coluna}_media_{nome}'] = grupos[coluna].transform(
            lambda serie: serie.rolling(amostras, min_periods=1).mean()
        )
        dados[f'{coluna}_desvio_{nome}'] = grupos[coluna].transform(
            lambda serie: serie.rolling(amostras, min_periods=2).std().fillna(0)
        )

dados['iem'] = dados['torque_fuso_nm'] * dados['vibracao_triaxial_rms_mms'] / (
    dados['vazao_refrigerante_lmin'] + 1
)

for coluna in dinamicas:
    media_liga = dados.groupby('liga_metalica')[coluna].transform('mean').replace(0, np.nan)
    dados[f'{coluna}_relativo_liga'] = dados[coluna] / media_liga

display(dados[dinamicas + ['iem']].describe().round(2))

,temperatura_fuso_c,vibracao_triaxial_rms_mms,torque_fuso_nm,vazao_refrigerante_lmin,iem
count,10000.00,10000.00,10000.00,10000.00,10000.00
mean,52.49,1.86,114.76,27.77,7.68
std,9.71,0.47,25.85,2.12,3.31
min,30.05,0.80,48.84,18.41,1.67
25%,45.05,1.53,96.13,26.41,5.21
50%,51.55,1.81,110.72,27.80,6.93
75%,58.56,2.14,130.42,29.21,9.58
max,74.98,3.80,208.97,35.68,25.26


## 7. Exportação
O CSV preparado é salvo em dois locais: `saida/` para o treinamento e `site/data/` para a dashboard.

In [7]:
arquivo_saida = PASTA_SAIDA / 'sensores_preparados.csv'
arquivo_site = PASTA_SITE / 'sensores_preparados.csv'
dados.to_csv(arquivo_saida, index=False)
dados.to_csv(arquivo_site, index=False)
print(f'Dados preparados: {arquivo_saida} ({len(dados):,} registros)')
print(f'Arquivo da dashboard: {arquivo_site}')

Dados preparados: C:\Users\gabriel_l_simao\Documents\GitHub\VulcanTech_Smart_Metals\saida\sensores_preparados.csv (10,000 registros)
Arquivo da dashboard: C:\Users\gabriel_l_simao\Documents\GitHub\VulcanTech_Smart_Metals\site\data\sensores_preparados.csv
